# S-05: TabICLv2 + Variant A (driver removal) + species-disaggregated livestock, 10-year transient CMIP6 trajectory

**Context**: live-discussion follow-up to S-03. TabICLv2 is a one-shot foundation model (not a recursive day-by-day rollout), so its 365-day S-03 horizon can be extended arbitrarily far without the compounding-error risk a tree/SARIMAX rollout carries. S-03's Variant A (driver removal) keeps only the 10 columns a CMIP6 scenario can actually supply (TA/SWIN/PRECIP/DOY/season/livestock) -- almost exactly what `data/Simulated Climate Data/` (the Semenov et al. CMIP6/LARS-WG dataset, already used by S-04) provides directly.

This combines three prior pieces of work rather than building a new pipeline:
- **S-03's Variant A** feature set (10 CMIP6-derivable columns).
- **F-10/D-67's species-disaggregated livestock density** (`fx_cattle_dens`/`fx_sheep_dens`/`fx_lamb_dens`) -- the same family behind the standing `TabPFN+species` forecasting champion.
- **S-04's transient scenario machinery** (real CMIP6 weather, realization-level sampling, AOA extrapolation flagging).

**Livestock multiplier -- user-confirmed Option B**: independent per-species multipliers (cattle/sheep/lamb each scaled separately, 1x/2x/3x per species = 27 combos), not S-01/S-04's single shared scalar. `fx_lsu_dens` is rebuilt as the exact LSU-weighted sum of the (scaled) species densities (`1.0*cattle + 0.1*sheep + 0.05*lamb`), preserving F-10's own construction identity under any combination.

**Scope** (cost measured empirically before committing, not guessed): 3 towers x 2 SSPs x 5 GCMs x 10 realizations/GCM (stratified, matching S-04's own precedent for cutting this exact axis) x 27 species-multiplier combos = 8,100 `tabicl_forecast()` calls, 10-year horizon each. Actual runtime: **2.54 hours**, 0 skipped/failed calls.

In [1]:
import sys, os, warnings
warnings.filterwarnings("ignore")
ROOT = r"c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project"
sys.path.insert(0, ROOT)
sys.path.insert(0, ROOT + r"\src")
sys.path.insert(0, ROOT + r"\src\features")
sys.path.insert(0, ROOT + r"\notebooks\07_scenario_analysis")

import pandas as pd
import numpy as np
pd.set_option("display.width", 220)

RESULTS = rf"{ROOT}\results"

df = pd.read_csv(f"{RESULTS}/s05_trajectory_realizations.csv")
print(f"s05_trajectory_realizations.csv: {len(df):,} rows")
for c in ["tower", "ssp", "gcm", "realization", "mult_cattle", "mult_sheep", "mult_lamb", "year"]:
    print(f"  {c}: {df[c].nunique()} unique")
assert df.isna().sum().sum() == 0, "unexpected NaNs in sweep output"

s05_trajectory_realizations.csv: 81,000 rows
  tower: 3 unique
  ssp: 2 unique
  gcm: 5 unique
  realization: 10 unique
  mult_cattle: 3 unique
  mult_sheep: 3 unique
  mult_lamb: 3 unique
  year: 14 unique


## Full sweep + analysis

The full 8,100-call sweep (`s05_trajectory_10yr.py`, ~2.54h) and its analysis (`s05_analysis.py`) were run via committed scripts, not re-executed here -- loading their already-saved outputs, matching the project's established script-does-the-heavy-lifting pattern (B-10/S-03/S-04).

In [2]:
import s05_analysis as s05a
s05a.main()

[OK] s05_trajectory_realizations.csv: 81,000 rows, year_offset range 1-10


[OK] s05_trajectory_summary.csv (1620 rows)
[OK] s05_realization_spread_pooled.csv (year+realization+GCM conflated, S-04-comparable but see caveat)
 tower    ssp      mean      std       p10       p90   n  band_pct_of_mean
     2 ssp245  5.497562 1.863733  3.448074  7.161211 500         67.541529
     2 ssp585  5.536362 1.873728  3.463699  7.265448 500         68.668736
     4 ssp245 13.194632 1.647582 10.870318 15.125004 500         32.245582
     4 ssp585 13.224065 1.669094 10.817841 15.178811 500         32.977533
     9 ssp245 20.348071 2.545745 15.963091 23.145868 500         35.299546
     9 ssp585 20.355535 2.566157 15.962760 23.205277 500         35.580090

[OK] s05_realization_spread_isolated.csv (realization+GCM ONLY, fixed year -- the actual apples-to-apples question)
 tower    ssp  realization_only_band_pct_of_mean
     2 ssp245                           5.696593
     2 ssp585                           6.607356
     4 ssp245                           2.354370
     4 ssp585 

[OK] figure: s05_trajectory_bands.png
[OK] figure: s05_aoa_trend.png


[OK] figure: s05_species_response.png

[DONE] S-05 analysis complete.


## 1. Species marginal response -- the core new question

Holding the other two species at 1x, how much does scaling cattle/sheep/lamb ALONE move predicted FCH4?

In [3]:
resp = pd.read_csv(f"{RESULTS}/s05_species_marginal_response.csv")
print(resp.pivot_table(index=["tower", "species"], columns="multiplier", values="pct_change_vs_baseline").round(1))

print("\nReal historical LSU-weight composition (why cattle might be expected to dominate):")
dv3 = pd.read_csv(f"{ROOT}/data/Hourly/forecast_daily_v3.csv", low_memory=False)
for t in [2, 4, 9]:
    sub = dv3[dv3.tower == t]
    c, s, l = sub["fx_cattle_dens"].mean(), sub["fx_sheep_dens"].mean(), sub["fx_lamb_dens"].mean()
    lsu = 1.0*c + 0.1*s + 0.05*l
    print(f"  Tower {t}: cattle {100*1.0*c/lsu:.0f}% of LSU, sheep {100*0.1*s/lsu:.0f}%, lamb {100*0.05*l/lsu:.0f}%")

multiplier     1.0    2.0    3.0
tower species                   
2     cattle   0.0    1.3    2.3
      lamb     0.0    4.0   10.0
      sheep    0.0    0.7    1.7
4     cattle   0.0  120.4  205.6
      lamb     0.0    3.5   10.0
      sheep    0.0    2.8    5.4
9     cattle   0.0   90.8  195.6
      lamb     0.0    5.5   13.6
      sheep    0.0    9.8   22.0

Real historical LSU-weight composition (why cattle might be expected to dominate):
  Tower 2: cattle 77% of LSU, sheep 7%, lamb 17%
  Tower 4: cattle 88% of LSU, sheep 5%, lamb 7%
  Tower 9: cattle 74% of LSU, sheep 16%, lamb 10%


**Cattle dominates far beyond its own LSU-weight share** at T4/T9 (tripling cattle alone roughly triples predicted FCH4; sheep/lamb stay under 25% even at 3x). Tower 2 is muted across every species -- consistent with its known different-regime status (D-18 lineage), not a new limitation.

## 2. Joint vs. additive -- do the three species interact?

In [4]:
joint = pd.read_csv(f"{RESULTS}/s05_joint_vs_additive.csv")
print(joint.round(2).to_string(index=False))

 tower  baseline  actual_joint_3x3x3  additive_prediction  cattle_delta  sheep_delta  lamb_delta  synergy_pct
     2      5.52                5.78                 6.29          0.13         0.09        0.55        -8.06
     4     13.21               42.32                42.41         27.16         0.72        1.31        -0.19
     9     20.35               73.34                67.42         39.81         4.49        2.77         8.77


T4 (best-covered tower) is almost exactly additive (-0.2% synergy). T9 shows a real +8.8% super-additive effect. T2's -8.1% is not read as real -- its absolute deltas are tiny (0.1-0.5 nmol), dominated by noise at that scale.

## 3. Realization/GCM spread -- a genuine correction was needed here

First pass (pooling year+realization+GCM together, matching S-04 Finding 1's own convention) gave a startling 32-69% band -- an order of magnitude larger than S-04's 1-5%. Investigated directly: a single fixed (GCM, realization, SSP) already ranges 9.98-15.46 across its own 10 years at T4 -- the pooled number was dominated by genuine year-to-year weather variability, not by which of the 50 weather sequences was drawn.

In [5]:
pooled = pd.read_csv(f"{RESULTS}/s05_realization_spread_pooled.csv")
isolated = pd.read_csv(f"{RESULTS}/s05_realization_spread_isolated.csv")
print("=== Pooled (year+realization+GCM conflated -- S-04-comparable but see caveat) ===")
print(pooled.to_string(index=False))
print("\n=== Isolated (realization+GCM ONLY, fixed year -- the actual apples-to-apples number) ===")
print(isolated.to_string(index=False))

=== Pooled (year+realization+GCM conflated -- S-04-comparable but see caveat) ===
 tower    ssp      mean      std       p10       p90   n  band_pct_of_mean
     2 ssp245  5.497562 1.863733  3.448074  7.161211 500         67.541529
     2 ssp585  5.536362 1.873728  3.463699  7.265448 500         68.668736
     4 ssp245 13.194632 1.647582 10.870318 15.125004 500         32.245582
     4 ssp585 13.224065 1.669094 10.817841 15.178811 500         32.977533
     9 ssp245 20.348071 2.545745 15.963091 23.145868 500         35.299546
     9 ssp585 20.355535 2.566157 15.962760 23.205277 500         35.580090

=== Isolated (realization+GCM ONLY, fixed year -- the actual apples-to-apples number) ===
 tower    ssp  realization_only_band_pct_of_mean
     2 ssp245                           5.696593
     2 ssp585                           6.607356
     4 ssp245                           2.354370
     4 ssp585                           2.602999
     9 ssp245                           5.193688
     9 s

Isolating the realization/GCM axis alone gives 2.4-6.6% -- consistent with S-04's own finding. The gap between the two numbers is a real, TabICLv2-specific finding: S-04's hybrid has a smooth Ridge trend that damps year-to-year weather noise; TabICLv2 has no such backbone and is directly, much more strongly sensitive to which specific year's weather it sees. Not a contradiction of S-04 -- a real architectural difference in scenario behavior between the two models.

## 4. AOA extrapolation risk -- high in absolute level, flat over the horizon

In [6]:
aoa = pd.read_csv(f"{RESULTS}/s05_aoa_trend.csv")
base = aoa[(aoa.mult_cattle == 1) & (aoa.mult_sheep == 1) & (aoa.mult_lamb == 1)]
print(base.groupby(["tower", "year_offset"])["aoa_flagged_pct"].mean().unstack("year_offset").round(1))

year_offset    1     2     3     4     5     6     7     8     9     10
tower                                                                  
2            68.0  67.8  67.9  68.2  68.6  68.0  68.2  68.0  68.3  68.1
4            62.8  62.6  62.6  62.3  63.0  62.8  62.9  63.0  62.6  63.1
9            68.3  67.9  68.1  67.7  68.1  67.9  67.7  68.4  67.8  68.0


62-68% flagged at every tower, essentially unchanged from year 1 to year 10 post-anchor -- no evidence extrapolation risk grows with horizon length, matching S-04's own Finding 3. The high absolute level (vs. S-04's 9-15%) is the concrete confirmation of S-04's own "AOA can be diluted by many in-range dimensions" hypothesis: S-05's feature space is deliberately narrow (13 columns, Variant A's whole point) vs. S-04's ~40+, so it dilutes less.

## 5. SSP2-4.5 vs SSP5-8.5 -- small, as everywhere else in this project

In [7]:
div = pd.read_csv(f"{RESULTS}/s05_ssp_divergence.csv")
print(div.to_string(index=False))

 tower      window    ssp245    ssp585  ssp585_pct_of_ssp245
     2 early_yr1_5  5.654460  5.697637              0.763595
     2 late_yr6_10  5.340665  5.375087              0.644520
     4 early_yr1_5 12.248489 12.259785              0.092221
     4 late_yr6_10 14.140776 14.188346              0.336407
     9 early_yr1_5 18.688607 18.704972              0.087565
     9 late_yr6_10 22.007535 22.006098             -0.006530


## Verdict

1. **F-10's species-split feature genuinely earns its place in a scenario context, not just on real historical anchors.** Cattle density is the dominant lever by a wide margin at T4/T9 -- more so than its own LSU-weight share would predict -- while sheep/lamb contribute comparatively little even at 3x. A digital-shadow interface built on the aggregate `fx_lsu_dens` alone would miss this asymmetry entirely.
2. **TabICLv2 is a viable long-horizon scenario forecaster**, and its one-shot (non-recursive) architecture is what makes a 10-year, 8,100-combination sweep tractable in ~2.5 hours.
3. **Realization/GCM-choice uncertainty is small once correctly isolated (2-7%)**, echoing S-04 -- but TabICLv2's year-to-year weather sensitivity (no smoothing trend component) is a real, separate source of variation, and conflating the two (as a naive S-04-style pooled metric would) overstates "realization spread" by an order of magnitude. Caught and corrected in this session, not left as a misleading headline number.
4. **AOA's absolute flagged-% depends heavily on feature-space breadth**, now confirmed a second time (S-04 -> S-05) under two different feature spaces -- any AOA number from this project should be read with its feature-space dimensionality stated alongside it.
5. **No change to the standing forecasting or scenario recommendations** -- this is a diagnostic extension of S-03/F-10, not a production-config change. See `s05_results.md` for the full write-up, caveats, and file inventory.

## Update: extended to 2050 + full daily chains (supersedes the 10-year results above)

User follow-up: (1) run the same 8,100-call grid to **2050** (matching S-04's own endpoint)
instead of a fixed 10 years -- T4/T9 now run 2024-2050 (27 years), T2 2020-2050 (31 years); (2)
save full daily chains for every call, not just the annual mean. Both done together in one pass
(`s05_trajectory_2050.py`) since the horizon extension, not the daily save, is what dominates the
new cost (measured: a single 27-year call takes 4.07s vs. ~1.2-1.3s for the original 10-year
call). Full grid estimated ~9h; **actual runtime 5.44h**, 0 failed calls. Daily output: 83,767,500
rows, written incrementally to Parquet (never held in memory at once) -- **1.25 GB**. Reproducibility
spot-checked directly: the same scenario point scored 9.981 in the original run and 9.974 here --
0.07% apart, ordinary GPU inference variance, not drift.

In [8]:
import s05_analysis_2050 as s05a2050
s05a2050.main()

[OK] s05_trajectory_realizations_2050.csv: 229,500 rows, year_offset range 1-31
       min  max
tower          
2        1   31
4        1   27
9        1   27


[OK] s05_trajectory_summary_2050.csv (4590 rows)
[OK] s05_realization_spread_pooled_2050.csv (year+realization+GCM conflated)
 tower    ssp      mean      std       p10       p90    n  band_pct_of_mean
     2 ssp245  5.779672 1.643477  3.528184  8.115999 1550         79.378471
     2 ssp585  5.881634 1.666949  3.583578  8.219532 1550         78.820855
     4 ssp245 14.232950 1.491423 12.189171 15.864213 1350         25.820660
     4 ssp585 14.303621 1.520714 12.264914 15.989378 1350         26.038607
     9 ssp245 21.736634 2.121181 19.394054 23.982655 1350         21.109985
     9 ssp585 21.657023 2.095686 19.358057 23.846311 1350         20.724242

[OK] s05_realization_spread_isolated_2050.csv (realization+GCM ONLY, fixed year)
 tower    ssp  realization_only_band_pct_of_mean
     2 ssp245                           6.296021
     2 ssp585                           7.560631
     4 ssp245                           2.521260
     4 ssp585                           2.857631
     9 ssp245  

[OK] figure: s05_trajectory_bands_2050.png
[OK] figure: s05_aoa_trend_2050.png


[OK] figure: s05_species_response_2050.png

[DONE] S-05 (2050 horizon) analysis complete.


## Updated verdict (2050 horizon)

Every finding from the 10-year version replicates, several more clearly:

1. **Cattle still dominates the species response**, and the magnitude holds up (T4 3x-alone: +205.6% -> +214.5%; T9: +195.6% -> +186.4%) -- not an artifact of the shorter original window.
2. **Joint-vs-additive holds**: T4 stays close to additive (-0.2% -> -0.6% synergy), T9's real super-additive effect persists (+8.8% -> +9.1%).
3. **Realization/GCM spread, isolated, stays in the same small range** (2.4-6.6% -> 2.5-7.6%) -- the pooled-vs-isolated distinction (Result 3 in `s05_results.md`) remains essential at the longer horizon too, since more years pooled means more year-to-year weather variability to conflate with realization choice if not separated correctly.
4. **AOA's flatness over time is now confirmed far more strongly** -- stable within ~1 percentage point across the *entire* 27-31-year horizon, not just 10 years.
5. **New at this horizon**: SSP2-4.5 vs SSP5-8.5 divergence now visibly grows from the early to the late window (matching S-04's own "widens toward end of century" pattern) -- the 10-year window was too short to show this; T9 shows a direction-inconsistent late-window number worth noting rather than smoothing over.

Still no change to any standing forecasting or scenario recommendation. Full detail, including the daily-resolution seasonal-pattern sanity check, in `s05_results.md`.

## Second update: farming-practice scenarios (grazing timing, fertilizer schedule)

User follow-up: extend beyond livestock density to two management-practice levers, run as two
**separate** experiments (not stacked onto the 27-combo livestock grid), each holding livestock at
baseline (1x/1x/1x) and sweeping only its own 3 levels, at the 2050 horizon.

**Priors stated before running (not fitted after the fact)**: both expected to show a smaller
effect than livestock's cattle result (F-01/F-04/F-05's "redundant on the rich base" finding for
management features), but the two axes were expected to differ from each other -- grazing timing
directly tied to livestock presence (the #1 driver everywhere in this project), fertilizer's
stronger mechanistic link is to N2O (not modeled here), not CH4.

**Construction**: grazing timing phase-shifts the real day-of-year species-density climatology at
the season edges (earlier turnout/later housing), re-deriving `fx_grazing_active`/
`fx_days_since_grazing` via `days_since_grazing()` (the exact function the real columns use,
reused unchanged). Fertilizer builds a "typical year" template from real per-tower event history
(T4: ~8.25 events/yr, DOY 82-234, mean 127 kg/ha; T9: ~4/yr; T2: ~5/yr), scaled by rate/frequency,
and run through `recency_series()` (the exact `exp(-days/14)` decay the real
`fx_mgmt_fertN_recency`/`_rate` columns use, also reused unchanged).

**Scope**: 900 calls/axis (3 towers x 2 SSPs x 5 GCMs x 10 realizations x 3 levels), smoke-tested
first. **Actual runtime: ~51 min/axis (~1.7h combined)**, 0 skipped/failed calls.

In [9]:
import s05_practices_analysis as s05p
s05p.main()

=== Grazing timing ===
[OK] s05_practices_grazing.csv: 25,500 rows
level  historical  plus2wk  plus4wk  pct_vs_historical_max
tower                                                     
2           7.432    7.551    7.735                  4.076
4          14.813   16.093   17.610                 18.880
9          30.665   33.116   35.930                 17.168

AOA-flagged %% by level:
level  historical  plus2wk  plus4wk
tower                              
2            85.4     88.1     88.1
4            76.0     82.0     88.0
9            78.4     83.8     87.2
[OK] figure: s05_practices_grazing.png

=== Fertilizer schedule ===
[OK] s05_practices_fertilizer.csv: 25,500 rows
level  historical  plus50pct_freq  plus50pct_rate  pct_vs_historical_max
tower                                                                   
2           5.030           4.823           4.989                 -0.814
4          13.383          13.550          13.503                  0.901
9          17.192        

### Practices verdict

Both priors confirmed, cleanly:

1. **Grazing timing shows a real, substantial, monotonic effect at every tower** -- T4: 14.81 -> 17.61 nmol (+18.9% at +4 weeks), T9: 30.67 -> 35.93 (+17.2%), T2 muted but same monotonic direction (+4.1%). A genuine, previously-untested management lever worth taking seriously alongside livestock density.
2. **Fertilizer schedule shows a small effect, inconsistent in DIRECTION across towers** -- T2/T9 show increased frequency *decreasing* predicted FCH4, T4 shows the opposite, every magnitude under 5%. Read as genuinely weak/noise-level, not a directional finding -- extends F-01/F-04/F-05's "redundant on the rich base" finding from real-data feature importance to scenario response.
3. **AOA side-finding**: grazing's AOA-flagged-% is both higher in absolute level and grows monotonically with the shift level (T4: 76.0%->88.0%) -- extending the season genuinely pushes the scenario further from the training distribution, a sensible pattern fertilizer doesn't show as cleanly.
4. **No change to any standing recommendation** -- both are diagnostic scenario extensions. Full detail in `s05_results.md`'s "Second update" section.